# Polygon API Exploration
Explore bars + financials endpoints before hardening into `PolygonClient`.

In [1]:
import os
from dotenv import load_dotenv
from polygon import RESTClient
import pandas as pd

load_dotenv()  # loads POLYGON_API_KEY from .env
client = RESTClient(api_key=os.environ['POLYGON_API_KEY'])
print('connected')

connected


## 1. Hourly Bars

In [2]:
# Fetch 1 month of hourly bars for AAPL
bars = []
for bar in client.list_aggs(
    ticker='AAPL',
    multiplier=1,
    timespan='hour',
    from_='2024-01-01',
    to='2024-02-01',
    adjusted=True,
    sort='asc',
    limit=50000,
):
    bars.append(bar)

print(f'{len(bars)} bars fetched')
print(vars(bars[0]))  # inspect raw fields

352 bars fetched
{'open': 191.52, 'high': 191.52, 'low': 189.8, 'close': 189.95, 'volume': 78491, 'vwap': 190.0364, 'timestamp': 1704186000000, 'transactions': 2918, 'otc': None}


In [3]:
# Convert to DataFrame
df = pd.DataFrame([vars(b) for b in bars])
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
df = df.set_index('timestamp')
df.head()

,open,high,low,close,volume,vwap,transactions,otc
timestamp,,,,,,,,
2024-01-02 09:00:00+00:00,191.52,191.52,189.80,189.95,78491.0,190.0364,2918,None
2024-01-02 10:00:00+00:00,190.00,190.24,189.70,189.76,29855.0,190.0020,1305,None
2024-01-02 11:00:00+00:00,189.68,189.68,188.80,188.94,121577.0,189.2501,3305,None
2024-01-02 12:00:00+00:00,189.02,189.65,188.40,188.40,174909.0,189.0329,4993,None
2024-01-02 13:00:00+00:00,189.03,192.53,187.78,188.10,555948.0,188.4356,16181,None


## 2. Financials (book value)

In [4]:
# Fetch annual financials for AAPL
financials = []
for f in client.vx.list_stock_financials(
    ticker='AAPL',
    timeframe='annual',
    limit=10,
):
    financials.append(f)

print(f'{len(financials)} filings')
print(vars(financials[0]))  # inspect top-level fields

17 filings
{'cik': '0000320193', 'company_name': 'Apple Inc.', 'end_date': '2025-09-27', 'filing_date': '2025-10-31', 'financials': Financials(balance_sheet=BalanceSheet(assets=DataPoint(label='Assets', order=100, unit='USD', value=359241000000.0, derived_from=None, formula=None, source=None, xpath=None), current_assets=DataPoint(label='Current Assets', order=200, unit='USD', value=147957000000.0, derived_from=None, formula=None, source=None, xpath=None), cash=DataPoint(label=None, order=None, unit=None, value=None, derived_from=None, formula=None, source=None, xpath=None), accounts_receivable=DataPoint(label=None, order=None, unit=None, value=None, derived_from=None, formula=None, source=None, xpath=None), inventory=DataPoint(label='Inventory', order=230, unit='USD', value=5718000000.0, derived_from=None, formula=None, source=None, xpath=None), prepaid_expenses=DataPoint(label=None, order=None, unit=None, value=None, derived_from=None, formula=None, source=None, xpath=None), other_cur

In [5]:
# Drill into balance sheet — looking for stockholders equity / book value
f0 = financials[0]
print('filing date:', f0.filing_date)
print('period:', f0.start_date, '->', f0.end_date)
print()
bs = f0.financials.balance_sheet
print(vars(bs))

filing date: 2025-10-31
period: 2024-09-29 -> 2025-09-27

{'assets': DataPoint(label='Assets', order=100, unit='USD', value=359241000000.0, derived_from=None, formula=None, source=None, xpath=None), 'current_assets': DataPoint(label='Current Assets', order=200, unit='USD', value=147957000000.0, derived_from=None, formula=None, source=None, xpath=None), 'cash': DataPoint(label=None, order=None, unit=None, value=None, derived_from=None, formula=None, source=None, xpath=None), 'accounts_receivable': DataPoint(label=None, order=None, unit=None, value=None, derived_from=None, formula=None, source=None, xpath=None), 'inventory': DataPoint(label='Inventory', order=230, unit='USD', value=5718000000.0, derived_from=None, formula=None, source=None, xpath=None), 'prepaid_expenses': DataPoint(label=None, order=None, unit=None, value=None, derived_from=None, formula=None, source=None, xpath=None), 'other_current_assets': DataPoint(label='Other Current Assets', order=250, unit='USD', value=142239000

In [6]:
# Extract equity — shares not in balance sheet, will use market_cap instead
for f in financials:
    bs = f.financials.balance_sheet
    equity = getattr(bs, 'equity', None)
    print(f.end_date, '| equity:', equity.value if equity and equity.value else None)

2025-09-27 | equity: 73733000000.0
2024-09-28 | equity: 56950000000.0
2023-09-30 | equity: 62146000000.0
2022-09-24 | equity: 50672000000.0
2021-09-25 | equity: 63090000000.0
2020-09-26 | equity: 65339000000.0
2019-09-28 | equity: 90488000000.0
2018-09-29 | equity: 107147000000.0
2017-09-30 | equity: 134047000000.0
2016-09-24 | equity: 128249000000.0
2015-09-26 | equity: 119355000000.0
2014-09-27 | equity: 111547000000.0
2013-09-28 | equity: 123549000000.0
2012-09-29 | equity: 118210000000.0
2011-09-24 | equity: 76615000000.0
2010-09-25 | equity: 47791000000.0
2009-09-26 | equity: 27832000000.0


## 3. Sanity check B/M for AAPL
B/M = book value per share / price at filing date

In [7]:
# Get market cap from ticker details on the filing date
# B/M = equity / market_cap (no shares needed)
f0 = financials[0]
bs = f0.financials.balance_sheet
equity = bs.equity.value

details = client.get_ticker_details('AAPL', date=f0.end_date)
print(vars(details))  # inspect what fields are available

{'active': True, 'address': CompanyAddress(address1='ONE APPLE PARK WAY', address2=None, city='CUPERTINO', state='CA', country=None, postal_code='95014'), 'branding': Branding(icon_url='https://api.polygon.io/v1/reference/company-branding/YXBwbGUuY29t/images/2025-04-04_icon.png', logo_url='https://api.polygon.io/v1/reference/company-branding/YXBwbGUuY29t/images/2025-04-04_logo.svg', accent_color=None, light_color=None, dark_color=None), 'cik': '0000320193', 'composite_figi': 'BBG000B9XRY4', 'currency_name': 'usd', 'currency_symbol': None, 'base_currency_name': None, 'base_currency_symbol': None, 'delisted_utc': None, 'description': "Apple is among the largest companies in the world, with a broad portfolio of hardware and software products targeted at consumers and businesses. Apple's iPhone makes up a majority of the firm sales, and Apple's other products like Mac, iPad, and Watch are designed around the iPhone as the focal point of an expansive software ecosystem. Apple has progressiv

In [8]:
# Compute B/M = equity / market_cap
market_cap = details.market_cap
bm = equity / market_cap if market_cap else None

print(f'Period end : {f0.end_date}')
print(f'Equity     : {equity:,.0f}')
print(f'Market cap : {market_cap:,.0f}' if market_cap else 'Market cap : None')
print(f'B/M        : {bm:.4f}' if bm else 'B/M        : None')

Period end : 2025-09-27
Equity     : 73,733,000,000
Market cap : 3,791,126,029,400
B/M        : 0.0194


## Universe exploration

## Universe EDA — moved to 02_universe_eda.ipynb

In [9]:
# list_tickers parameters worth knowing:
#   market   : 'stocks' | 'crypto' | 'fx' | 'otc'
#   exchange : 'XNAS' (Nasdaq) | 'XNYS' (NYSE) | 'XASE' (AMEX)
#   type     : 'CS' (common stock) | 'ETF' | 'ADRC' (ADR) etc.
#   active   : True = currently trading

tickers = []
for t in client.list_tickers(
    market='stocks',
    exchange='XNYS',   # NYSE only to keep it manageable
    type='CS',         # common stock only
    active=True,
    limit=1000,
):
    tickers.append(t)

print(f'{len(tickers)} tickers')
print(vars(tickers[0]))  # inspect available fields

1737 tickers
{'ticker': 'A', 'name': 'Agilent Technologies Inc.', 'market': 'stocks', 'locale': 'us', 'primary_exchange': 'XNYS', 'type': 'CS', 'active': True, 'currency_name': 'usd', 'cik': '0001090872', 'composite_figi': 'BBG000C2V3D6', 'share_class_figi': 'BBG001SCTQY4', 'last_updated_utc': '2026-03-27T06:08:03.745778795Z'}


In [10]:
# Convert ticker list to DataFrame
ticker_df = pd.DataFrame([vars(t) for t in tickers])
print(ticker_df.shape)
print(ticker_df.dtypes)
ticker_df.head()

(1737, 12)
ticker               str
name                 str
market               str
locale               str
primary_exchange     str
type                 str
active              bool
currency_name        str
cik                  str
composite_figi       str
share_class_figi     str
last_updated_utc     str
dtype: object


,ticker,name,market,locale,primary_exchange,type,active,currency_name,cik,composite_figi,share_class_figi,last_updated_utc
0,A,Agilent Technologies Inc.,stocks,us,XNYS,CS,True,usd,0001090872,BBG000C2V3D6,BBG001SCTQY4,2026-03-27T06:08:03.745778795Z
1,AA,Alcoa Corporation,stocks,us,XNYS,CS,True,usd,0001675149,BBG00B3T3HD3,BBG00B3T3HF1,2026-03-27T06:08:03.745779346Z
2,AAMI,Acadian Asset Management Inc.,stocks,us,XNYS,CS,True,usd,0001748824,BBG00P2HLNY3,BBG00P2HLNZ2,2026-03-27T06:08:03.74578155Z
3,AAP,ADVANCE AUTO PARTS INC,stocks,us,XNYS,CS,True,usd,0001158449,BBG000F7RCJ1,BBG001SD2SB2,2026-03-27T06:08:03.745781891Z
4,AAT,"AMERICAN ASSETS TRUST, INC.",stocks,us,XNYS,CS,True,usd,0001500217,BBG00161BCR0,BBG001TCBJS5,2026-03-27T06:08:03.745783304Z


## 4. Test PolygonClient

In [11]:
import sys
sys.path.insert(0, '..')
from algotrading.db.polygon_client import PolygonClient

pc = PolygonClient()

# Single ticker
df = pc.get_bars('AAPL', '2024-01-01', '2024-02-01')
print(df.shape)
df.head()

(352, 8)


,ticker,open,high,low,close,volume,vwap,transactions
timestamp,,,,,,,,
2024-01-02 09:00:00+00:00,AAPL,191.52,191.52,189.80,189.95,78491.0,190.0364,2918
2024-01-02 10:00:00+00:00,AAPL,190.00,190.24,189.70,189.76,29855.0,190.0020,1305
2024-01-02 11:00:00+00:00,AAPL,189.68,189.68,188.80,188.94,121577.0,189.2501,3305
2024-01-02 12:00:00+00:00,AAPL,189.02,189.65,188.40,188.40,174909.0,189.0329,4993
2024-01-02 13:00:00+00:00,AAPL,189.03,192.53,187.78,188.10,555948.0,188.4356,16181


In [12]:
# Bulk — 3 tickers, threaded
df_bulk = pc.get_bars_bulk(['AAPL', 'MSFT', 'GOOG'], '2024-01-01', '2024-02-01')
print(df_bulk.shape)
print(df_bulk['ticker'].value_counts())

(1056, 8)
ticker
AAPL    352
MSFT    352
GOOG    352
Name: count, dtype: int64


In [13]:
# Universe dim table
dim = pc.get_tickers()
print(dim.shape)
dim.head(10)

(5028, 7)


,ticker,name,primary_exchange,type,active,cik,currency_name
0,A,Agilent Technologies Inc.,XNYS,CS,True,0001090872,usd
1,AA,Alcoa Corporation,XNYS,CS,True,0001675149,usd
2,AAMI,Acadian Asset Management Inc.,XNYS,CS,True,0001748824,usd
3,AAP,ADVANCE AUTO PARTS INC,XNYS,CS,True,0001158449,usd
4,AAT,"AMERICAN ASSETS TRUST, INC.",XNYS,CS,True,0001500217,usd
5,AAUC,Allied Gold Corporation,XNYS,CS,True,0001993344,usd
6,AB,"AllianceBernstein Holding, L.P.",XNYS,CS,True,0000825313,usd
7,ABBV,ABBVIE INC.,XNYS,CS,True,0001551152,usd
8,ABCB,Ameris Bancorp,XNYS,CS,True,0000351569,usd
9,ABG,"Asbury Automotive Group, Inc.",XNYS,CS,True,0001144980,usd


## 5. Write dim table to S3

In [14]:
from algotrading.db.s3_client import S3Client

# prefix='dev' — writes to dev/ folder, never touches prod/
s3 = S3Client(prefix='dev')

In [15]:
# Read back and verify
dim_back = s3.read_dim()
print(dim_back.shape)
dim_back.head()

(0, 0)


""


## 6. Fetch + write price bars to S3

In [16]:
# Small test: 5 tickers, 1 month
test_tickers = ['AAPL', 'MSFT', 'GOOG', 'JPM', 'XOM']
bars = pc.get_bars_bulk(test_tickers, '2024-01-01', '2024-02-01')
print(bars.shape)
print(bars['ticker'].value_counts())
bars.head()

(1720, 8)
ticker
AAPL    352
MSFT    352
GOOG    352
XOM     345
JPM     319
Name: count, dtype: int64


,ticker,open,high,low,close,volume,vwap,transactions
timestamp,,,,,,,,
2024-01-02 09:00:00+00:00,AAPL,191.52,191.52,189.80,189.95,78491.0,190.0364,2918
2024-01-02 09:00:00+00:00,MSFT,376.73,376.81,376.29,376.29,1730.0,376.6188,50
2024-01-02 09:00:00+00:00,GOOG,140.93,141.05,140.80,140.97,3528.0,140.8999,90
2024-01-02 09:00:00+00:00,JPM,170.05,170.19,170.00,170.00,3475.0,170.0720,75
2024-01-02 09:00:00+00:00,XOM,100.77,101.00,100.28,100.90,20327.0,100.8414,351


In [17]:
# Write to dev/bars/hourly/...
s3.write_bars(bars)
print('done')

done


In [18]:
# Read back and verify round-trip
bars_back = s3.read_bars('2024-01-01', '2024-02-01')
print(bars_back.shape)
print(bars_back['ticker'].value_counts())
bars_back.head()

(1641, 8)
ticker
AAPL    336
MSFT    336
GOOG    336
XOM     329
JPM     304
Name: count, dtype: int64


,ticker,open,high,low,close,volume,vwap,transactions
timestamp,,,,,,,,
2024-01-02 09:00:00+00:00,AAPL,191.52,191.52,189.80,189.95,78491.0,190.0364,2918
2024-01-02 09:00:00+00:00,MSFT,376.73,376.81,376.29,376.29,1730.0,376.6188,50
2024-01-02 09:00:00+00:00,GOOG,140.93,141.05,140.80,140.97,3528.0,140.8999,90
2024-01-02 09:00:00+00:00,JPM,170.05,170.19,170.00,170.00,3475.0,170.0720,75
2024-01-02 09:00:00+00:00,XOM,100.77,101.00,100.28,100.90,20327.0,100.8414,351
